![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

https://www.quantconnect.com/datasets/quantconnect-forex

In [1]:
from QuantConnect import *
from QuantConnect.Research import QuantBook
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

qb = QuantBook()

start = pd.Timestamp(2022, 5, 1, tz="UTC") 
end = pd.Timestamp(2023, 5, 1, tz="UTC")  

# Tier 1 forex pairs for BTC prediction
tier1_pairs = ['USDJPY', 'EURUSD', 'GBPUSD', 'AUDUSD', 'USDCNH']

# Dictionary to store all forex data
forex_data = {}

print("Fetching Tier 1 Forex Data...")
print("=" * 50)

for pair in tier1_pairs:
    try:
        print(f"\nFetching {pair}...")
        
        # Add forex pair
        symbol = qb.add_forex(pair, Resolution.MINUTE).symbol
        
        # Get minute data
        history = qb.history(symbol, start, end, Resolution.MINUTE, fill_forward=True)
        
        if history.empty:
            print(f"  ✗ No data for {pair}")
            continue
        
        # Reset index and resample to 5-min
        history = history.reset_index()
        history['time_5min'] = history['time'].dt.floor('5min')
        
        # Aggregate to 5-minute bars
        forex_5min = history.groupby('time_5min').agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'bidclose': 'last',
            'askclose': 'last'
        })
        
        # Store in dictionary
        forex_data[pair] = forex_5min
        
        print(f"  ✓ {len(forex_5min)} bars ({forex_5min.index.min()} to {forex_5min.index.max()})")
        
    except Exception as e:
        print(f"  ✗ Error fetching {pair}: {str(e)}")

print("\n" + "=" * 50)
print(f"Successfully fetched {len(forex_data)} out of {len(tier1_pairs)} pairs")

# Optional: Display summary of each pair
if forex_data:
    print("\nData Summary:")
    for pair, data in forex_data.items():
        print(f"{pair}: {len(data)} bars, Close range: {data['close'].min():.4f} - {data['close'].max():.4f}")

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

def add_forex_features(df, ticker):
    """
    Stationary features for forex pairs as BTC predictors.
    All features prefixed with forex_{ticker}_. No raw OHLCV in output.
    
    Focus: USD strength, risk sentiment, carry dynamics, and bid-ask spreads
    """
    features = df.copy()
    
    # ============================================================
    # PRICE MOMENTUM (Returns - stationary)
    # ============================================================
    features['ret_1'] = features['close'].pct_change(1)      # 5-min
    features['ret_3'] = features['close'].pct_change(3)      # 15-min
    features['ret_12'] = features['close'].pct_change(12)    # 1-hour
    features['ret_48'] = features['close'].pct_change(48)    # 4-hour
    features['ret_288'] = features['close'].pct_change(288)  # 1-day
    
    # Momentum acceleration (second derivative)
    features['ret_accel_12'] = features['ret_12'] - features['ret_12'].shift(12)
    features['ret_accel_48'] = features['ret_48'] - features['ret_48'].shift(48)
    
    # Momentum percentile ranks (risk-on/risk-off signals)
    features['ret_pctrank_12'] = features['ret_12'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    features['ret_pctrank_48'] = features['ret_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # BID-ASK SPREAD (Liquidity & volatility proxy)
    # ============================================================
    if 'bidclose' in features.columns and 'askclose' in features.columns:
        # Spread in basis points
        mid_price = (features['bidclose'] + features['askclose']) / 2
        features['spread_bps'] = ((features['askclose'] - features['bidclose']) / mid_price) * 10000
        
        # Spread relative to recent average (liquidity stress indicator)
        spread_ma_12 = features['spread_bps'].rolling(12).mean()
        spread_ma_48 = features['spread_bps'].rolling(48).mean()
        
        features['spread_ratio_1h'] = features['spread_bps'] / (spread_ma_12 + 1e-8)
        features['spread_ratio_4h'] = features['spread_bps'] / (spread_ma_48 + 1e-8)
        
        # Spread volatility (market stress)
        features['spread_vol_12'] = features['spread_bps'].rolling(12).std()
        features['spread_vol_48'] = features['spread_bps'].rolling(48).std()
        
        # Spread percentile (high spread = low liquidity)
        features['spread_pctrank_48'] = features['spread_bps'].rolling(48).apply(
            lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
        )
    
    # ============================================================
    # VOLATILITY (Realized vol - risk sentiment proxy)
    # ============================================================
    features['rvol_12'] = features['ret_1'].rolling(12).std()   # 1-hour
    features['rvol_48'] = features['ret_1'].rolling(48).std()   # 4-hour
    features['rvol_288'] = features['ret_1'].rolling(288).std() # 1-day
    
    # Volatility ratio (recent vs longer-term)
    features['vol_ratio_12_48'] = features['rvol_12'] / (features['rvol_48'] + 1e-8)
    features['vol_ratio_48_288'] = features['rvol_48'] / (features['rvol_288'] + 1e-8)
    
    # Volatility percentile (high vol = high uncertainty)
    features['vol_pctrank_48'] = features['rvol_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # Volatility regime change
    features['vol_regime_change'] = (features['rvol_48'] - features['rvol_48'].shift(48)) / (features['rvol_288'] + 1e-8)
    
    # ============================================================
    # RANGE-BASED VOLATILITY (Parkinson estimator)
    # ============================================================
    if 'high' in features.columns and 'low' in features.columns:
        # Parkinson volatility (more efficient than close-to-close)
        log_hl = np.log(features['high'] / features['low'])
        features['parkinson_vol_12'] = np.sqrt((log_hl ** 2).rolling(12).mean() / (4 * np.log(2)))
        features['parkinson_vol_48'] = np.sqrt((log_hl ** 2).rolling(48).mean() / (4 * np.log(2)))
        
        # Range as % of price
        features['range_pct'] = (features['high'] - features['low']) / features['close']
        features['range_pct_ma12'] = features['range_pct'].rolling(12).mean()
        
        # Range percentile
        features['range_pctrank_48'] = features['range_pct'].rolling(48).apply(
            lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
        )
    
    # ============================================================
    # MOMENTUM PERSISTENCE (Trend strength)
    # ============================================================
    # Count of positive returns (directional consistency)
    features['positive_ret_ratio_12'] = features['ret_1'].rolling(12).apply(lambda x: (x > 0).sum() / len(x))
    features['positive_ret_ratio_48'] = features['ret_1'].rolling(48).apply(lambda x: (x > 0).sum() / len(x))
    
    # Return skewness (asymmetry in moves)
    features['ret_skew_48'] = features['ret_1'].rolling(48).skew()
    features['ret_skew_288'] = features['ret_1'].rolling(288).skew()
    
    # Return kurtosis (tail risk)
    features['ret_kurt_48'] = features['ret_1'].rolling(48).kurt()
    features['ret_kurt_288'] = features['ret_1'].rolling(288).kurt()
    
    # ============================================================
    # MEAN REVERSION SIGNALS
    # ============================================================
    # Z-score of price relative to moving average
    sma_48 = features['close'].rolling(48).mean()
    sma_288 = features['close'].rolling(288).mean()
    std_48 = features['close'].rolling(48).std()
    std_288 = features['close'].rolling(288).std()
    
    features['zscore_48'] = (features['close'] - sma_48) / (std_48 + 1e-8)
    features['zscore_288'] = (features['close'] - sma_288) / (std_288 + 1e-8)
    
    # Moving average crossover signal
    features['ma_cross_48_288'] = (sma_48 - sma_288) / features['close']
    
    # ============================================================
    # SEASONALITY & CYCLICAL PATTERNS
    # ============================================================
    # Return vs same time yesterday
    features['ret_vs_1d_ago'] = features['close'].pct_change(288) / (features['ret_288'].rolling(288).std() + 1e-8)
    
    # Price percentile within day
    features['price_pctrank_1d'] = features['close'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # VOLUME-LIKE PROXIES (if volume available - rare in spot forex)
    # ============================================================
    if 'volume' in features.columns:
        features['vol_ret_12'] = features['volume'].pct_change(12)
        features['vol_ret_48'] = features['volume'].pct_change(48)
        
        vma_48 = features['volume'].rolling(48).mean()
        features['rvol_ratio'] = features['volume'] / (vma_48 + 1e-8)
    
    # ============================================================
    # DROP RAW OHLCV (keep only derived features)
    # ============================================================
    features = features.drop(['open', 'high', 'low', 'close', 'bidclose', 'askclose', 'volume'], 
                             axis=1, errors='ignore')
    
    # ============================================================
    # PREFIX ALL COLUMNS WITH forex_{ticker}_
    # ============================================================
    rename_dict = {col: f'forex_{ticker.lower()}_{col}' for col in features.columns}
    features = features.rename(columns=rename_dict)
    
    return features


# Apply to all forex pairs
print("="*60)
print("CREATING FOREX FEATURES (STATIONARY ONLY)")
print("="*60)

forex_features = {}

for ticker, df in forex_data.items():
    print(f"\n{ticker}...", end=" ")
    features = add_forex_features(df.copy(), ticker)
    forex_features[ticker] = features
    print(f"✓ {features.shape[1]} features, {features.shape[0]} rows")

print("\n" + "="*60)
print(f"Total forex pairs processed: {len(forex_features)}")

In [3]:
# Merge all forex features together
print("Merging all forex features...")

# Start with first forex pair
tickers = list(forex_features.keys())
merged_forex = forex_features[tickers[0]].copy()
print(f"  ✓ Starting with {tickers[0]}: {merged_forex.shape[1]} features")

# Merge remaining pairs
for ticker in tickers[1:]:
    merged_forex = merged_forex.join(forex_features[ticker], how='inner')
    print(f"  ✓ Merged {ticker}: {forex_features[ticker].shape[1]} features")

print(f"\nMerged forex dataset: {merged_forex.shape[0]} rows × {merged_forex.shape[1]} columns")

Forex is very liquid, after removing warmup period, no Nans.

In [4]:
# Find and drop warmup period
first_complete_idx = merged_forex.dropna().index[0]
warmup_idx = merged_forex.index.get_loc(first_complete_idx)

print(f"Dropping first {warmup_idx} rows (warmup period)...")
merged_forex_clean = merged_forex.iloc[warmup_idx:].copy()

# Check remaining NaNs
remaining_nans = merged_forex_clean.isna().sum().sum()
print(f"Remaining NaNs: {remaining_nans}")

if remaining_nans > 0:
    nan_cols = merged_forex_clean.isna().sum()
    nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
    print("\nColumns with NaNs:")
    for col, count in nan_cols.head(10).items():
        print(f"  - {col}: {count} ({count/len(merged_forex_clean)*100:.2f}%)")

In [5]:
# Save merged forex dataset
merged_forex_clean.to_parquet('forex_merged.parquet')
print(f"✓ Saved forex_merged.parquet")
print(f"  Shape: {merged_forex_clean.shape}")
print(f"  Date range: {merged_forex_clean.index.min()} to {merged_forex_clean.index.max()}")